In [ ]:
!pip install pyspark
import pyspark
print(pyspark.__version__)

In [ ]:
KAFKA_VERSION = "4.3.1"
SCALA_VERSION = "2.13"
KAFKA_DIR = f"kafka_{SCALA_VERSION}-{KAFKA_VERSION}"

!wget https://downloads.apache.org/kafka/{KAFKA_VERSION}/{KAFKA_DIR}.tgz
!tar -xzf {KAFKA_DIR}.tgz

In [ ]:
import subprocess

config_file = f"{KAFKA_DIR}/config/server.properties"

# Generar Cluster ID limpio
salida_uuid = subprocess.check_output(
    [f"{KAFKA_DIR}/bin/kafka-storage.sh", "random-uuid"]
).decode().strip()

cluster_id = salida_uuid.splitlines()[-1].strip()
print(f"Cluster ID limpio: '{cluster_id}'")

# Formatear almacenamiento en modo standalone (KRaft)
!rm -rf /tmp/kraft-combined-logs
!{KAFKA_DIR}/bin/kafka-storage.sh format --standalone -t {cluster_id} -c {config_file}

In [ ]:
import time
import subprocess

# Liberar puerto 9092 si quedó ocupado por ejecuciones previas
!fuser -k 9092/tcp 2>/dev/null || true

kafka_log = open("kafka_server.log", "w")
kafka_process = subprocess.Popen(
    [
        f"{KAFKA_DIR}/bin/kafka-server-start.sh",
        config_file
    ],
    stdout=kafka_log,
    stderr=subprocess.STDOUT
)

time.sleep(15)

if kafka_process.poll() is None:
    print(f"Kafka corriendo exitosamente con PID {kafka_process.pid}")
else:
    print("Kafka se detuvo:")
    !tail -n 30 kafka_server.log

In [ ]:
if kafka_process.poll() is None:
    print("Kafka server esta corriendo con PID", kafka_process.pid)
else:
    print("Kafka se detuvo -log:")
    !tail -n 30 kafka_server.log

In [ ]:
!{KAFKA_DIR}/bin/kafka-topics.sh --create --topic viajes \
--bootstrap-server localhost:9092 \
--replication-factor 1 --partitions 1

In [ ]:
!{KAFKA_DIR}/bin/kafka-topics.sh --list --bootstrap-server localhost:9092

In [ ]:
!pip install kafka-python-ng

In [ ]:
import json
import random
import time
import threading
from datetime import datetime
from kafka import KafkaProducer

producer = KafkaProducer(
    bootstrap_servers="localhost:9092",
    value_serializer=lambda v: json.dumps(v).encode("utf-8")
)

ciudades = ["Bogotá", "Medellín", "Cali", "Tunja", "Barranquilla"]

def generar_viajes():
    return {
        "ciudad": random.choice(ciudades),
        "tarifa": round(random.uniform(10000, 50000), 0),
        "duracion_min": random.randint(120, 780),
        "timestamp": datetime.now().isoformat()
    }

def producir_continuamente():
    while True:
        producer.send("viajes", generar_viajes())
        producer.flush()
        time.sleep(0.3)

hilo_productor = threading.Thread(target=producir_continuamente, daemon=True)
hilo_productor.start()
print("Productor corriendo en segundo plano y enviando datos sin buffering")

In [ ]:
from pyspark.sql import SparkSession
import pyspark

spark_version = pyspark.__version__
kafka_package = f"org.apache.spark:spark-sql-kafka-0-10_{SCALA_VERSION}:{spark_version}"
print("Usando el conector: ", kafka_package)

spark = SparkSession.builder \
    .appName("Viajes") \
    .config("spark.jars.packages", kafka_package) \
    .getOrCreate()

print("SparkSession lista")

In [ ]:
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, IntegerType
from pyspark.sql.functions import from_json, col, count, avg, round

esquema = StructType() \
    .add("ciudad", StringType()) \
    .add("tarifa", DoubleType()) \
    .add("duracion_min", IntegerType()) \
    .add("timestamp", StringType())

kafka_df = spark.readStream \
    .format("kafka") \
    .option("kafka.bootstrap.servers", "localhost:9092") \
    .option("subscribe", "viajes") \
    .option("startingOffsets", "earliest") \
    .load()

viajes_df = kafka_df.select(
    from_json(col("value").cast("string"), esquema).alias("data")
).select("data.*")

print("Stream de viajes listo para agregarse")

In [ ]:
# Detener cualquier streaming query previo para evitar conflictos o saturación en memoria
for s in spark.streams.active:
    s.stop()

resumen = viajes_df.groupBy("ciudad") \
    .agg(count("*").alias("total_viajes"), \
         round(avg("tarifa"), 2).alias("tarifa_promedio"))

query = resumen.writeStream \
    .outputMode("complete") \
    .format("memory") \
    .queryName("demanda_ciudades") \
    .trigger(processingTime="2 seconds") \
    .start()

print("Consulta de streaming iniciada correctamente")

In [ ]:
import time

print("⏳ Esperando a que Spark procese el primer lote de Kafka (Batch 0)...")
# Esperar hasta que el micro-batch 0 se procese y la tabla en memoria tenga registros
while True:
    try:
        total = spark.sql("SELECT count(*) FROM demanda_ciudades").collect()[0][0]
        if total > 0:
            break
    except Exception:
        pass
    time.sleep(1)

print("🚀 ¡Primer lote procesado con éxito! Monitoreando 6 actualizaciones en tiempo real:\n")

for i in range(6):
    print(f" -- Actualización {i+1} de 6 -- ")
    spark.sql("SELECT ciudad, total_viajes, tarifa_promedio FROM demanda_ciudades ORDER BY total_viajes DESC").show()
    if i < 5:
        time.sleep(5)
